In [2]:
import os
import mne
import pyedflib
import numpy as np
import scipy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import jupyterlab
import yasa
import glob
import gc
import warnings


# Configuration for interactive plots
%matplotlib widget

# Safe import of psutil with error handling in case it is not installed
try:    
    import psutil
except ImportError:    
    psutil = None

# Definition of the RAM check function
def print_ram(label=""):
    if psutil is None:
        print(f"{label} RAM: unavailable (psutil not installed)")
        return
        
    process = psutil.Process(os.getpid())
    mb = process.memory_info().rss / 1024 / 1024
    print(f"{label} RAM: {mb:.1f} MB")


print_ram("Initial state")

Initial state RAM: 454.8 MB


In [3]:
import config
from config import DATA_FOLDER

# Retrieve a list of paths to all .edf files in that folder
edf_files = sorted(DATA_FOLDER.glob("*.edf"))

# Display the found files and their total count
print(f"Found {len(edf_files)} EDF files.")
for file in edf_files:
    print(file)

Found 3 EDF files.
/home/olivia/eeg_data/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data.edf
/home/olivia/eeg_data/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{891F8634-3CC8-4FEE-B1D3-64394DC86F7A} Data.edf
/home/olivia/eeg_data/___EEG/CZD_sampleData/CZD_sampleData/eeg_edf/2023-07-27{D29553B9-1220-4DFC-8456-F712EE2281C8} Data.edf


In [4]:
def create_bipolar(raw_input):

    electrodes = {}

    for ch in raw_input.ch_names:
        if not ch[-1].isdigit():
            continue

        electrode = ch.rstrip("0123456789")

        if electrode not in electrodes:
            electrodes[electrode] = []

        electrodes[electrode].append(ch)

    anode = []
    cathode = []
    bipolar_names = []

    for electrode in electrodes:
        # Sort channels by their actual number (integer value)
        channels = sorted(
            electrodes[electrode],
            key=lambda x: int(x[len(electrode):])
        )

        for i in range(len(channels) - 1):
            # Extract numerical numbers for the current and next channel
            current_num = int(channels[i][len(electrode):])
            next_num = int(channels[i + 1][len(electrode):])
            
            # CONDITION: Create a pair only if they are adjacent (the difference is exactly 1)
            if next_num - current_num == 1:
                anode.append(channels[i])
                cathode.append(channels[i + 1])
                bipolar_names.append(f"{channels[i]}-{channels[i + 1]}")
            else:
                # Optional: you can add a print statement here, e.g., print(f"Skipped gap: {channels[i]} to {channels[i+1]}")
                continue

    raw_bipolar = mne.set_bipolar_reference(
        raw_input,
        anode=anode,
        cathode=cathode,
        ch_name=bipolar_names,
        drop_refs=False,
        copy=True,
        verbose=False
    )
 
    return raw_bipolar, bipolar_names

In [10]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal
import mne

# 1. Enable Matplotlib interactive mode
plt.ion()

# 2. Silence MNE info logging (removes "Plotting power spectral density...")
mne.set_log_level('WARNING')

# Loop over EDF files
for edf_file in edf_files:
    file_base_name = os.path.splitext(os.path.basename(edf_file))[0]
    
    # Print start marker for file processing
    print("\n" + "="*130)
    print(f'PROCESSING THIS FILE: {file_base_name} .......................')
    print("="*130)
    
    output_dir = f"SOZ/{file_base_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    # Load raw EDF metadata silently using latin1 encoding
    raw = mne.io.read_raw_edf(edf_file, preload=False, encoding="latin1", verbose=False)

    # Extract annotations table
    annotations = pd.DataFrame({
        "onset_s": raw.annotations.onset,
        "duration_s": raw.annotations.duration,
        "description": raw.annotations.description
    })

    # Save full inventory table to CSV
    annotations_path = f"{output_dir}/inventory_table_annotations.csv"
    annotations.to_csv(annotations_path, index=False)
    print(f"[INFO] Saved full inventory table to: {annotations_path}")

    # Search for seizure markers ("NAPAD")
    seizure_rows = annotations[annotations["description"].str.contains("NAPAD", na=False)]

    if seizure_rows.empty:
        print(f"[RESULT] No seizure ('NAPAD') found in this file. Skipping.")
        continue  

    total_seizures = len(seizure_rows)
    print(f"[RESULT] Found {total_seizures} seizure event(s) ('NAPAD') in this file.")

    # Process each seizure event individually
    for index, row in enumerate(seizure_rows.itertuples(), start=1):
        onset = row.onset_s
        tmin = max(0, onset - 60)
        tmax = onset + 30

        seizure_output_dir = f"{output_dir}/seizure_{index}"
        os.makedirs(seizure_output_dir, exist_ok=True)
        os.makedirs(f"{seizure_output_dir}/spectrograms", exist_ok=True)

        # Crop and load signal segment
        raw_cropped = raw.copy().crop(tmin=tmin, tmax=tmax, verbose=False)
        raw_cropped.load_data(verbose=False)

        # Apply notch filter and bandpass filter (1-250 Hz)
        raw_filtered = raw_cropped.copy()
        raw_filtered.notch_filter(freqs=[50, 100, 150, 200, 250], fir_design='firwin', verbose=False)
        raw_filtered.filter(l_freq=1.0, h_freq=250.0, fir_design='firwin', verbose=False)

        # Construct bipolar montage
        raw_bipolar_filtered, bipolar_names = create_bipolar(raw_filtered)
        total_channels = len(bipolar_names)

        print(f"  -> Seizure #{index}/{total_seizures} | Onset: {onset:.1f}s | Window: {tmin:.1f}s to {tmax:.1f}s | Channels filtered: {total_channels}")

        # ==============================================================================
        # 1. BIPOLAR TRACE PLOT (SAVE STATIC PNG)
        # ==============================================================================
        # A. Save static PNG trace image via Matplotlib backend
        mne.viz.set_browser_backend('matplotlib')
        fig_filtered_save = raw_bipolar_filtered.plot(
            picks=bipolar_names, n_channels=55, duration=10.0, scalings="auto",
            title=f"FILTERED Bipolar trace - Seizure {index} | {file_base_name}", show=False, verbose=False
        )
        fig_filtered_save.savefig(f"{seizure_output_dir}/bipolar_filtered_trace.png", dpi=300)
        plt.close(fig_filtered_save)

        # B. Display non-blocking interactive Qt browser window !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        
        mne.viz.set_browser_backend('qt')
        fig_filtered_interactive = raw_bipolar_filtered.plot(
            picks=bipolar_names, n_channels=55, duration=10.0, scalings="auto",
            title=f"FILTERED Bipolar trace - Seizure {index} | {file_base_name}", show=True, block=False, verbose=False
        )
        

        # ==============================================================================
        # 2. COMPUTE PSD & AUTOMATIC FLAT/DEAD CHANNEL DETECTION
        # ==============================================================================
        print(f"     Computing PSD for Seizure #{index}...")
        psd = raw_bipolar_filtered.compute_psd(
            fmin=0,
            fmax=40,
            picks=bipolar_names,
            verbose='ERROR'
        )

        # Identify flat or disconnected channels based on average spectral power
        psd_data = psd.get_data()
        mean_power = np.mean(psd_data, axis=1)
        dead_indices = np.where(mean_power < 1e-20)[0]
        
        if len(dead_indices) > 0:
            dead_channels = [bipolar_names[i] for i in dead_indices]
            print(f"     [WARNING] Detected {len(dead_channels)} disconnected/flat channel(s): {', '.join(dead_channels)}")
            
            # Exclude flat channels from channel list
            bipolar_names = [ch for ch in bipolar_names if ch not in dead_channels]
            total_channels = len(bipolar_names)
            
            # Re-compute PSD without flat channels
            psd = raw_bipolar_filtered.compute_psd(
                fmin=0,
                fmax=40,
                picks=bipolar_names,
                verbose='ERROR'
            )

        # ==============================================================================
        # 3. PSD PLOT (SAVE STATIC PNG)
        # ==============================================================================
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*Channel locations not available.*")
            
            # A. Save static PSD figure to disk
            fig_psd_save = psd.plot(average=False, show=False)
            fig_psd_save.suptitle(f"PSD - bipolar channels - Seizure {index} (Cleaned)", fontsize=14)
            fig_psd_save.savefig(
                f"{seizure_output_dir}/psd_bipolar.png",
                dpi=300,
                bbox_inches="tight"    
            )   
            plt.close(fig_psd_save)

            # B. Display non-blocking interactive PSD window (COMMENTED OUT)
            """
            fig_psd_interactive = psd.plot(average=False, show=False)
            fig_psd_interactive.suptitle(f"PSD - bipolar channels - Seizure {index} (Cleaned)", fontsize=14)
            fig_psd_interactive.show()
            """
        
        print(f"     [SUCCESS] Saved trace PNG and clean PSD graph PNG.")

        # ==============================================================================
        # 4. SPECTROGRAM GENERATION (COMMENTED OUT)
        # ==============================================================================
        """
        sfreq = raw_bipolar_filtered.info["sfreq"]
        for channel_for_spectrogram in bipolar_names:
            signal = raw_bipolar_filtered.get_data(picks=[channel_for_spectrogram], verbose=False)[0]
            f, t, Sxx = scipy.signal.spectrogram(signal, fs=sfreq, nperseg=512, noverlap=256)

            fig_spec = plt.figure(figsize=(10, 5))
            plt.pcolormesh(t, f, 10 * np.log10(Sxx), shading="gouraud")
            plt.ylim(0, 100)
            plt.xlabel("Time [s]")
            plt.ylabel("Frequency [Hz]")
            plt.title(f"Spectrogram - {channel_for_spectrogram} (Seizure {index})")
            plt.colorbar(label="Power [dB]")
            plt.tight_layout()

            fig_spec.savefig(f"{seizure_output_dir}/spectrograms/spectrogram_{channel_for_spectrogram}.png", dpi=300)
            plt.close(fig_spec)
        """

        # Memory cleanup
        del raw_cropped, raw_filtered, raw_bipolar_filtered, psd
        plt.close('all')
        gc.collect()

    print(f"\n[DONE] Finished processing file: {file_base_name}")


PROCESSING THIS FILE: 2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data .......................
[INFO] Saved full inventory table to: SOZ/2023-07-26{6591E658-D0E0-49FE-BDE4-FC101EACF45D} Data/inventory_table_annotations.csv
[RESULT] Found 6 seizure event(s) ('NAPAD') in this file.
  -> Seizure #1/6 | Onset: 15886.7s | Window: 15826.7s to 15916.7s | Channels filtered: 103
     Computing PSD for Seizure #1...
     [SUCCESS] Saved trace PNG and clean PSD graph PNG.
  -> Seizure #2/6 | Onset: 19941.6s | Window: 19881.6s to 19971.6s | Channels filtered: 103
     Computing PSD for Seizure #2...
     [SUCCESS] Saved trace PNG and clean PSD graph PNG.
  -> Seizure #3/6 | Onset: 20373.8s | Window: 20313.8s to 20403.8s | Channels filtered: 103
     Computing PSD for Seizure #3...
     [WARNING] Detected 2 disconnected/flat channel(s): RM6-RM7, RM7-RM8
     [SUCCESS] Saved trace PNG and clean PSD graph PNG.
  -> Seizure #4/6 | Onset: 31834.5s | Window: 31774.5s to 31864.5s | Channels filtere